# BSD Sector Analysis

Headline dynamism figures by sector, plus a systematic anomaly list for microdata follow-up.

**Data:** `29_02_2026_BSD_Dynamism_Stats.xlsx`  
**Measures available at sector level:** Entry & exit rates · Job reallocation · DHS growth rates · HGF/stagnant shares · Productivity dispersion · Firm/employment counts  
**Not available at sector level:** Survival (cohort sheet has no sector dimension)

---

## Setup

In [1]:
import pandas as pd
import numpy as np
import altair as alt
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import sys
sys.path.append('.')
from eco_style import pallete
alt.themes.enable('report')
alt.data_transformers.enable('default', max_rows=None)

script_dir = Path.cwd()
DATA = script_dir.parent / 'Data' / 'BSD' / '29_02_2026_BSD_Dynamism_Stats.xlsx'

xl   = pd.ExcelFile(DATA)
pop  = xl.parse('population')
fd   = xl.parse('firm_dynamics')
jf   = xl.parse('job_flows')
gr   = xl.parse('growth_rates')
gc   = xl.parse('growth_cats')
prod = xl.parse('prod')

def to_numeric(df):
    df = df.copy()
    for col in df.columns:
        df[col] = pd.to_numeric(df[col].replace('*', np.nan), errors='ignore')
    return df

pop  = to_numeric(pop)
fd   = to_numeric(fd)
jf   = to_numeric(jf)
gr   = to_numeric(gr)
gc   = to_numeric(gc)
prod = to_numeric(prod)

gc['year'] = gc['year'].astype('Int64')
gc = gc.dropna(subset=['year'])

# Exclude 1998 (first year — rates undefined due to lagged denominator)
fd   = fd[fd['year']   != 1998].copy()
jf   = jf[jf['year']   != 1998].copy()
gc   = gc[gc['year']   != 1998].copy()

# Sector slices
fd_s   = fd[fd['dimension']   == 'Sector'].copy()
jf_s   = jf[jf['dimension']   == 'Sector'].copy()
gr_s   = gr[gr['dimension']   == 'Sector'].copy()
gc_s   = gc[gc['dimension']   == 'Sector'].copy()
prod_s = prod[prod['dimension'] == 'Sector'].copy()
pop_s  = pop[pop['dimension']  == 'Sector'].copy()

SECTORS = sorted(fd_s['category'].unique())

# Colour scale: one colour per sector, cycling through nominal palette
PALETTE_COLS = [pallete[k] for k in sorted(pallete) if k.startswith('nominal')]
SECTOR_COLORS = (PALETTE_COLS * 5)[:len(SECTORS)]

PERIODS = {
    'Pre-GFC (1999–2007)':      (1999, 2007),
    'GFC (2008–2010)':          (2008, 2010),
    'Recovery (2011–2019)':     (2011, 2019),
    'COVID & post (2020–2022)': (2020, 2022),
}

def period_means(df, cols):
    rows = []
    for label, (y0, y1) in PERIODS.items():
        sub = df[(df['year'] >= y0) & (df['year'] <= y1)]
        row = {'Period': label}
        for c in cols:
            row[c] = sub.groupby('category')[c].mean()
        rows.append(pd.DataFrame(row))
    return pd.concat(rows).reset_index().rename(columns={'category': 'Sector'})

print(f'Loaded. {len(SECTORS)} sectors: {SECTORS}')

Loaded. 15 sectors: ['Agriculture', 'Business Support Services', 'Construction', 'Hospitality', 'IT & Computer Services', 'Manufacturing', 'Other Information Services', 'Other Primary Industries', 'Other Services', 'Professional Services', 'Retail Trade', 'Social care', 'Transport & Logistics', 'Utilities', 'Wholesale Trade']


---
## 1. Entry & Exit Rates

Entry rate = (n_entrants + n_entry_and_exit) / n_firms_{t−1} × 100  
Exit rate = (n_exiters + n_entry_and_exit) / n_firms_{t−1} × 100

In [2]:
fd_s = fd_s.sort_values(['category', 'year'])
fd_s['n_firms_lag'] = fd_s.groupby('category')['n_firms'].shift(1)
fd_s['entry_rate']  = (fd_s['n_entrants'] + fd_s['n_entry_and_exit']) / fd_s['n_firms_lag'] * 100
fd_s['exit_rate']   = (fd_s['n_exiters']  + fd_s['n_entry_and_exit']) / fd_s['n_firms_lag'] * 100
fd_s['net_entry']   = fd_s['entry_rate'] - fd_s['exit_rate']
fd_s['churn_rate']  = fd_s['entry_rate'] + fd_s['exit_rate']
fd_s = fd_s.dropna(subset=['n_firms_lag'])

In [4]:
# Cross-sector comparison: average entry and exit rates (full period)
ee_avg = fd_s.groupby('category')[['entry_rate', 'exit_rate', 'churn_rate', 'net_entry']].mean().round(1).reset_index()
ee_avg = ee_avg.rename(columns={'category': 'Sector', 'entry_rate': 'Avg entry (%)',
                                 'exit_rate': 'Avg exit (%)', 'churn_rate': 'Avg churn (%)',
                                 'net_entry': 'Avg net entry (pp)'})
ee_avg.sort_values('Avg entry (%)', ascending=False)

,Sector,Avg entry (%),Avg exit (%),Avg churn (%),Avg net entry (pp)
13,Utilities,22.5,13.9,36.4,8.6
1,Business Support Services,19.2,17.7,36.9,1.5
12,Transport & Logistics,17.6,16.6,34.1,1.0
3,Hospitality,17.3,17.4,34.7,-0.1
4,IT & Computer Services,16.7,16.8,33.5,-0.1
9,Professional Services,15.5,13.9,29.4,1.6
6,Other Information Services,14.8,13.5,28.4,1.3
2,Construction,13.9,13.2,27.0,0.7
10,Retail Trade,12.6,13.4,26.1,-0.8
8,Other Services,10.7,11.2,21.8,-0.5


In [5]:
# Diverging bar: avg entry vs exit
ee_bar = fd_s.groupby('category')[['entry_rate', 'exit_rate']].mean().reset_index()
ee_bar_long = ee_bar.melt(id_vars='category', var_name='measure', value_name='rate')
ee_bar_long['measure'] = ee_bar_long['measure'].map({'entry_rate': 'Entry', 'exit_rate': 'Exit'})

alt.Chart(ee_bar_long).mark_bar().encode(
    x=alt.X('rate:Q', title='Average rate (%)'),
    y=alt.Y('category:N', sort='-x', title=''),
    color=alt.Color('measure:N',
        scale=alt.Scale(domain=['Entry', 'Exit'],
                        range=[pallete['nominal_1'], pallete['nominal_2']]),
        legend=alt.Legend(title='')),
    xOffset='measure:N',
    tooltip=['category:N', 'measure:N', alt.Tooltip('rate:Q', format='.1f')]
).properties(title='Average entry and exit rates by sector (1999–2022)', width=500, height=380)

alt.Chart(...)

---
## 2. Job Reallocation

JRR = (JC_entrants + JC_incumbents + JD_exiters + JD_incumbents) / emp_{t−1} × 100  
Extensive margin = (JC_entrants + JD_exiters) / emp_{t−1} × 100  
Intensive margin = (JC_incumbents + JD_incumbents) / emp_{t−1} × 100

In [6]:
jf_s = jf_s.sort_values(['category', 'year'])
jf_s['emp_lag']       = jf_s.groupby('category')['employment'].shift(1)
jf_s['jrr']           = (jf_s['job_creation_entrants'] + jf_s['job_creation_incumbents']
                        + jf_s['job_destruction_exiters'] + jf_s['job_destruction_incumbents']) / jf_s['emp_lag'] * 100
jf_s['jrr_extensive'] = (jf_s['job_creation_entrants']   + jf_s['job_destruction_exiters'])   / jf_s['emp_lag'] * 100
jf_s['jrr_intensive'] = (jf_s['job_creation_incumbents'] + jf_s['job_destruction_incumbents']) / jf_s['emp_lag'] * 100
jf_s['jcr']           = (jf_s['job_creation_entrants']   + jf_s['job_creation_incumbents'])   / jf_s['emp_lag'] * 100
jf_s['jdr']           = (jf_s['job_destruction_exiters'] + jf_s['job_destruction_incumbents']) / jf_s['emp_lag'] * 100
jf_s = jf_s.dropna(subset=['emp_lag'])

In [7]:
# JRR over time by sector (interactive)
sec_sel2 = alt.binding_select(options=SECTORS, name='Sector: ')
sec_par2 = alt.selection_point(fields=['category'], bind=sec_sel2, value=SECTORS[0])

jrr_lines = jf_s.melt(
    id_vars=['year', 'category'],
    value_vars=['jrr_extensive', 'jrr_intensive'],
    var_name='margin', value_name='rate'
)
jrr_lines['margin'] = jrr_lines['margin'].map({
    'jrr_extensive': 'Extensive (entry/exit)',
    'jrr_intensive': 'Intensive (incumbents)'
})

alt.Chart(jrr_lines).mark_area(opacity=0.7).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('rate:Q', title='% of t−1 employment', stack='zero'),
    color=alt.Color('margin:N',
        scale=alt.Scale(domain=['Extensive (entry/exit)', 'Intensive (incumbents)'],
                        range=[pallete['nominal_1'], pallete['nominal_3']]),
        legend=alt.Legend(title='')),
    tooltip=['year:O', 'margin:N', alt.Tooltip('rate:Q', format='.1f')]
).add_params(sec_par2).transform_filter(sec_par2
).properties(title='Job reallocation rate by sector — extensive vs intensive margins', width=640, height=300)

alt.Chart(...)

In [8]:
# Summary table
jrr_avg = jf_s.groupby('category')[['jrr', 'jrr_extensive', 'jrr_intensive', 'jcr', 'jdr']].mean().round(1).reset_index()
jrr_avg = jrr_avg.rename(columns={
    'category': 'Sector', 'jrr': 'JRR (%)', 'jrr_extensive': 'Extensive (%)',
    'jrr_intensive': 'Intensive (%)', 'jcr': 'JCR (%)', 'jdr': 'JDR (%)'
})
jrr_avg.sort_values('JRR (%)', ascending=False)

,Sector,JRR (%),Extensive (%),Intensive (%),JCR (%),JDR (%)
4,IT & Computer Services,31.2,15.0,16.2,17.2,14.0
1,Business Support Services,29.8,10.7,19.1,15.7,14.1
3,Hospitality,26.7,12.1,14.6,14.1,12.6
2,Construction,26.4,11.6,14.8,13.9,12.5
9,Professional Services,25.9,11.2,14.7,14.0,12.0
7,Other Primary Industries,23.0,7.5,15.5,10.3,12.7
8,Other Services,23.0,9.4,13.6,12.2,10.8
13,Utilities,22.5,7.7,14.8,12.4,10.2
6,Other Information Services,22.1,8.0,14.1,11.3,10.8
11,Social care,20.1,8.3,11.9,11.0,9.2


---
## 3. DHS Growth Rates

DHS growth rate: g = (E_t − E_{t−1}) / ((E_t + E_{t−1}) / 2), bounded [−2, 2]. Incumbents only.

In [9]:
# Mean DHS growth by sector — line chart (interactive)
sec_sel3 = alt.binding_select(options=SECTORS, name='Sector: ')
sec_par3 = alt.selection_point(fields=['category'], bind=sec_sel3, value=SECTORS[0])

gr_lines = gr_s.melt(
    id_vars=['year', 'category'],
    value_vars=['mean_dhs_growth', 'p10_dhs_growth', 'p90_dhs_growth'],
    var_name='stat', value_name='value'
)
gr_lines['stat'] = gr_lines['stat'].map({
    'mean_dhs_growth': 'Mean', 'p10_dhs_growth': 'P10', 'p90_dhs_growth': 'P90'
})

alt.Chart(gr_lines).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('value:Q', title='DHS growth rate'),
    color=alt.Color('stat:N',
        scale=alt.Scale(domain=['Mean', 'P10', 'P90'],
                        range=[pallete['nominal_1'], pallete['nominal_2'], pallete['nominal_3']]),
        legend=alt.Legend(title='')),
    tooltip=['year:O', 'stat:N', alt.Tooltip('value:Q', format='.3f')]
).add_params(sec_par3).transform_filter(sec_par3
).properties(title='DHS growth distribution by sector (mean, P10, P90)', width=640, height=300)

alt.Chart(...)

In [10]:
# Cross-sector averages
gr_avg = gr_s.groupby('category')[['mean_dhs_growth', 'median_dhs_growth', 'sd_dhs_growth', 'p90_dhs_growth']].mean().round(3).reset_index()
gr_avg = gr_avg.rename(columns={
    'category': 'Sector', 'mean_dhs_growth': 'Mean', 'median_dhs_growth': 'Median',
    'sd_dhs_growth': 'Std dev', 'p90_dhs_growth': 'P90'
})
gr_avg.sort_values('Mean', ascending=False)

,Sector,Mean,Median,Std dev,P90
13,Utilities,0.043,0.0,0.356,0.384
11,Social care,0.036,0.0,0.266,0.288
2,Construction,0.022,0.0,0.295,0.238
12,Transport & Logistics,0.022,0.0,0.283,0.264
1,Business Support Services,0.020,0.0,0.308,0.298
4,IT & Computer Services,0.019,0.0,0.272,0.187
10,Retail Trade,0.019,0.0,0.281,0.290
14,Wholesale Trade,0.019,0.0,0.268,0.224
9,Professional Services,0.016,0.0,0.275,0.202
5,Manufacturing,0.015,0.0,0.266,0.214


---
## 4. High-Growth and Stagnant Firms

HGF share = n_hgf / n_incumbents × 100  
Stagnant share = n_stagnant / n_incumbents × 100

In [11]:
gc_s = gc_s.copy()
gc_s['hgf_sh']  = gc_s['n_hgf']      / gc_s['n_incumbents'] * 100
gc_s['stag_sh'] = gc_s['n_stagnant'] / gc_s['n_incumbents'] * 100
gc_s['shrk_sh'] = gc_s['n_shrinkng'] / gc_s['n_incumbents'] * 100

In [12]:
# HGF and stagnant shares over time (interactive)
sec_sel4 = alt.binding_select(options=SECTORS, name='Sector: ')
sec_par4 = alt.selection_point(fields=['category'], bind=sec_sel4, value=SECTORS[0])

gc_long = gc_s.melt(
    id_vars=['year', 'category'],
    value_vars=['hgf_sh', 'stag_sh', 'shrk_sh'],
    var_name='type', value_name='share'
)
gc_long['type'] = gc_long['type'].map({'hgf_sh': 'High-growth', 'stag_sh': 'Stagnant', 'shrk_sh': 'Shrinking'})

alt.Chart(gc_long).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('share:Q', title='Share of incumbents (%)'),
    color=alt.Color('type:N',
        scale=alt.Scale(domain=['High-growth', 'Stagnant', 'Shrinking'],
                        range=[pallete['nominal_1'], pallete['nominal_3'], pallete['nominal_2']]),
        legend=alt.Legend(title='')),
    tooltip=['year:O', 'type:N', alt.Tooltip('share:Q', format='.1f')]
).add_params(sec_par4).transform_filter(sec_par4
).properties(title='HGF, stagnant and shrinking shares by sector', width=640, height=300)

alt.Chart(...)

In [13]:
# Cross-sector bar: avg HGF share vs stagnant share
gc_avg = gc_s.groupby('category')[['hgf_sh', 'stag_sh', 'shrk_sh']].mean().round(1).reset_index()

gc_bar = gc_avg.melt(id_vars='category', var_name='type', value_name='share')
gc_bar['type'] = gc_bar['type'].map({'hgf_sh': 'High-growth', 'stag_sh': 'Stagnant', 'shrk_sh': 'Shrinking'})

alt.Chart(gc_bar).mark_bar().encode(
    x=alt.X('share:Q', title='Share of incumbents (%)'),
    y=alt.Y('category:N', sort=alt.EncodingSortField(field='share', op='sum', order='descending'), title=''),
    color=alt.Color('type:N',
        scale=alt.Scale(domain=['High-growth', 'Stagnant', 'Shrinking'],
                        range=[pallete['nominal_1'], pallete['nominal_3'], pallete['nominal_2']]),
        legend=alt.Legend(title='')),
    xOffset='type:N',
    tooltip=['category:N', 'type:N', alt.Tooltip('share:Q', format='.1f')]
).properties(title='Average HGF, stagnant and shrinking shares by sector (1999–2022)', width=500, height=380)

alt.Chart(...)

---
## 5. Productivity

Turnover per employee (£ nominal). Dispersion measured by P90/P10 and P75/P25 ratios.

In [14]:
prod_s = prod_s.copy()
prod_s['p90_p10'] = prod_s['p90_prod'] / prod_s['p10_prod']
prod_s['p75_p25'] = prod_s['p75_prod'] / prod_s['p25_prod']

In [15]:
# Median productivity over time by sector (interactive)
sec_sel5 = alt.binding_select(options=SECTORS, name='Sector: ')
sec_par5 = alt.selection_point(fields=['category'], bind=sec_sel5, value=SECTORS[0])

alt.Chart(prod_s).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('p50_prod:Q', title='Median turnover/employee (£)', scale=alt.Scale(zero=False)),
    color=alt.Color('category:N', legend=None),
    tooltip=['year:O', 'category:N', alt.Tooltip('p50_prod:Q', format=',.0f')]
).add_params(sec_par5).transform_filter(sec_par5
).properties(title='Median productivity (turnover/employee) by sector', width=640, height=280)

alt.Chart(...)

In [16]:
# P90/P10 dispersion over time by sector (interactive)
sec_sel6 = alt.binding_select(options=SECTORS, name='Sector: ')
sec_par6 = alt.selection_point(fields=['category'], bind=sec_sel6, value=SECTORS[0])

alt.Chart(prod_s).mark_line(point=True, strokeWidth=2, color=pallete['nominal_2']).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('p90_p10:Q', title='P90/P10 ratio', scale=alt.Scale(zero=False)),
    tooltip=['year:O', 'category:N', alt.Tooltip('p90_p10:Q', format='.1f')]
).add_params(sec_par6).transform_filter(sec_par6
).properties(title='Productivity dispersion (P90/P10) by sector', width=640, height=280)

alt.Chart(...)

In [17]:
# Latest year cross-section
latest_prod = prod_s[prod_s['year'] == prod_s['year'].max()].copy()
latest_prod[['category', 'p10_prod', 'p50_prod', 'p90_prod', 'p90_p10']].rename(columns={
    'category': 'Sector', 'p10_prod': 'P10 (£)', 'p50_prod': 'Median (£)',
    'p90_prod': 'P90 (£)', 'p90_p10': 'P90/P10'
}).sort_values('Median (£)', ascending=False).style.format({
    'P10 (£)': '{:,.0f}', 'Median (£)': '{:,.0f}', 'P90 (£)': '{:,.0f}', 'P90/P10': '{:.1f}'
})

,Sector,P10 (£),Median (£),P90 (£),P90/P10
1106,Utilities,22,121,306,13.9
1133,Wholesale Trade,27,105,464,17.2
782,Business Support Services,18,100,213,11.8
809,Construction,30,98,276,9.2
917,Other Information Services,16,90,250,15.6
1025,Retail Trade,21,89,250,11.9
863,IT & Computer Services,16,77,151,9.4
890,Manufacturing,21,75,189,9.0
998,Professional Services,20,74,170,8.5
944,Other Primary Industries,11,73,220,20.0


---
## 6. Firm Population & Employment

Sector share of total firms and employment over time.

In [18]:
total_pop = pop[pop['dimension'] == 'Total'][['year', 'n_firms', 'employment']].rename(
    columns={'n_firms': 'total_firms', 'employment': 'total_emp'}
)
pop_s2 = pop_s.merge(total_pop, on='year')
pop_s2['firm_share'] = pop_s2['n_firms']    / pop_s2['total_firms'] * 100
pop_s2['emp_share']  = pop_s2['employment'] / pop_s2['total_emp']   * 100

# Employment share by sector over time
alt.Chart(pop_s2).mark_area().encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('emp_share:Q', title='Share of employment (%)', stack='normalize'),
    color=alt.Color('category:N', legend=alt.Legend(title='', columns=2)),
    tooltip=['year:O', 'category:N', alt.Tooltip('emp_share:Q', format='.1f')]
).properties(title='Sector composition of employment (share)', width=680, height=320)

alt.Chart(...)

In [19]:
# Average firm size by sector (latest year)
latest_pop = pop_s[pop_s['year'] == pop_s['year'].max()].copy()
latest_pop['avg_size'] = latest_pop['employment'] / latest_pop['n_firms']
latest_pop[['category', 'n_firms', 'employment', 'avg_size']].rename(columns={
    'category': 'Sector', 'n_firms': 'Firms', 'employment': 'Employment', 'avg_size': 'Avg size'
}).sort_values('Avg size', ascending=False).style.format({
    'Firms': '{:,.0f}', 'Employment': '{:,.0f}', 'Avg size': '{:.1f}'
})

,Sector,Firms,Employment,Avg size
1071,Social care,"51,986","1,738,533",33.4
1073,Utilities,"16,043","335,944",20.9
1065,Manufacturing,"141,602","2,487,562",17.6
1063,Hospitality,"178,198","2,535,031",14.2
1070,Retail Trade,"219,873","3,012,282",13.7
1061,Business Support Services,"236,832","2,969,640",12.5
1072,Transport & Logistics,"130,068","1,475,664",11.3
1074,Wholesale Trade,"188,932","1,817,239",9.6
1066,Other Information Services,"56,810","544,732",9.6
1067,Other Primary Industries,"10,928","87,385",8.0


---
## 7. Sector Scorecard

All key headline figures in one table.

In [20]:
scorecard = (
    fd_s.groupby('category')[['entry_rate', 'exit_rate', 'churn_rate']].mean()
    .join(jf_s.groupby('category')[['jrr', 'jrr_extensive', 'jrr_intensive']].mean())
    .join(gr_s.groupby('category')[['mean_dhs_growth', 'sd_dhs_growth']].mean())
    .join(gc_s.groupby('category')[['hgf_sh', 'stag_sh']].mean())
    .join(prod_s.groupby('category')[['p90_p10', 'p50_prod']].mean())
).round(2)

scorecard.index.name = 'Sector'
scorecard.columns = [
    'Avg entry (%)', 'Avg exit (%)', 'Avg churn (%)',
    'Avg JRR (%)', 'JRR ext (%)', 'JRR int (%)',
    'Mean DHS', 'SD DHS',
    'HGF share (%)', 'Stagnant share (%)',
    'P90/P10', 'Median prod (£)'
]

scorecard.style.background_gradient(cmap='RdYlGn', subset=[
    'Avg entry (%)', 'Avg JRR (%)', 'Mean DHS', 'HGF share (%)'
]).background_gradient(cmap='RdYlGn_r', subset=[
    'Avg exit (%)', 'Stagnant share (%)'
]).format({
    'Avg entry (%)': '{:.1f}', 'Avg exit (%)': '{:.1f}', 'Avg churn (%)': '{:.1f}',
    'Avg JRR (%)': '{:.1f}', 'JRR ext (%)': '{:.1f}', 'JRR int (%)': '{:.1f}',
    'Mean DHS': '{:.3f}', 'SD DHS': '{:.3f}',
    'HGF share (%)': '{:.1f}', 'Stagnant share (%)': '{:.1f}',
    'P90/P10': '{:.1f}', 'Median prod (£)': '{:,.0f}'
})

,Avg entry (%),Avg exit (%),Avg churn (%),Avg JRR (%),JRR ext (%),JRR int (%),Mean DHS,SD DHS,HGF share (%),Stagnant share (%),P90/P10,Median prod (£)
Sector,,,,,,,,,,,,
Agriculture,3.1,5.7,8.8,18.3,5.5,12.8,0.000,0.230,7.7,78.9,inf,46
Business Support Services,19.2,17.7,36.9,29.8,10.7,19.1,0.020,0.310,10.7,61.2,9.6,74
Construction,13.8,13.2,27.0,26.4,11.6,14.8,0.020,0.300,9.7,68.2,10.9,75
Hospitality,17.3,17.4,34.6,26.7,12.2,14.6,0.010,0.320,14.3,48.1,4.0,34
IT & Computer Services,16.7,16.8,33.5,31.2,15.0,16.2,0.020,0.270,8.3,65.9,6.3,63
Manufacturing,9.4,10.8,20.2,16.4,5.6,10.8,0.010,0.270,9.9,64.3,7.4,61
Other Information Services,14.8,13.5,28.4,22.1,8.0,14.1,0.020,0.300,8.7,67.2,13.1,69
Other Primary Industries,9.5,10.7,20.1,23.0,7.5,15.5,0.010,0.280,8.5,72.1,23.9,57
Other Services,10.7,11.2,21.8,23.0,9.4,13.6,0.010,0.280,10.7,65.2,7.8,37


---
## 8. Anomaly Detection

Systematic scan for data points to investigate in the microdata. Three complementary tests:

1. **Implausible values** — rates outside hard bounds (e.g. entry > 100%, JRR > 200%)
2. **Within-sector outliers** — year-on-year changes > 3 SD from each sector's own change distribution
3. **Cross-sector outliers** — sector-year values > 3 SD from the cross-sector distribution in each year

Each test produces a structured table; all anomalies are merged at the end.

In [21]:
# ── Test 1: Hard-bound violations ────────────────────────────────────────────

BOUNDS = [
    # (dataframe, measure_col, label, lo, hi)
    (fd_s, 'entry_rate',  'Entry rate (%)',  0,   100),
    (fd_s, 'exit_rate',   'Exit rate (%)',   0,   100),
    (fd_s, 'churn_rate',  'Churn rate (%)',  0,   200),
    (jf_s, 'jrr',         'JRR (%)',         0,   200),
    (jf_s, 'jcr',         'JCR (%)',         0,   200),
    (jf_s, 'jdr',         'JDR (%)',         0,   200),
    (gc_s, 'hgf_sh',      'HGF share (%)',   0,   100),
    (gc_s, 'stag_sh',     'Stagnant share (%)' , 0, 100),
]

bound_anomalies = []
for df, col, label, lo, hi in BOUNDS:
    mask = df[col].notna() & ((df[col] < lo) | (df[col] > hi))
    hits = df[mask][['year', 'category', col]].copy()
    hits['measure'] = label
    hits['value']   = hits[col].round(2)
    hits['anomaly_type'] = 'Hard-bound violation'
    hits['detail'] = hits.apply(
        lambda r: f"{r[col]:.2f} outside [{lo}, {hi}]", axis=1
    )
    bound_anomalies.append(hits[['year', 'category', 'measure', 'value', 'anomaly_type', 'detail']])

bound_df = pd.concat(bound_anomalies, ignore_index=True) if bound_anomalies else pd.DataFrame()
print(f'Hard-bound violations: {len(bound_df)}')
bound_df

Hard-bound violations: 1


,year,category,measure,value,anomaly_type,detail
0,2003,Utilities,Entry rate (%),123.87,Hard-bound violation,"123.87 outside [0, 100]"


In [22]:
# ── Test 2: Within-sector year-on-year outliers (|Δ| > 3 SD) ─────────────────

YOY_MEASURES = [
    (fd_s, 'entry_rate',      'Entry rate (%)'),
    (fd_s, 'exit_rate',       'Exit rate (%)'),
    (jf_s, 'jrr',             'JRR (%)'),
    (jf_s, 'jcr',             'JCR (%)'),
    (jf_s, 'jdr',             'JDR (%)'),
    (gr_s, 'mean_dhs_growth', 'Mean DHS growth'),
    (gc_s, 'hgf_sh',          'HGF share (%)'),
    (gc_s, 'stag_sh',         'Stagnant share (%)'),
    (prod_s, 'p50_prod',      'Median productivity (£)'),
    (prod_s, 'p90_p10',       'P90/P10 dispersion'),
]

yoy_anomalies = []
for df, col, label in YOY_MEASURES:
    tmp = df[['year', 'category', col]].dropna().sort_values(['category', 'year']).copy()
    tmp['delta'] = tmp.groupby('category')[col].diff()
    stats = tmp.groupby('category')['delta'].agg(['mean', 'std']).rename(
        columns={'mean': 'delta_mean', 'std': 'delta_std'}
    )
    tmp = tmp.merge(stats, on='category')
    tmp['z'] = (tmp['delta'] - tmp['delta_mean']) / tmp['delta_std']
    hits = tmp[tmp['z'].abs() > 3].copy()
    if len(hits):
        hits['measure'] = label
        hits['value']   = hits[col].round(3)
        hits['anomaly_type'] = 'Within-sector YoY outlier (|z|>3)'
        hits['detail'] = hits.apply(
            lambda r: f"Δ={r['delta']:+.3f}, z={r['z']:+.1f}", axis=1
        )
        yoy_anomalies.append(hits[['year', 'category', 'measure', 'value', 'anomaly_type', 'detail']])

yoy_df = pd.concat(yoy_anomalies, ignore_index=True) if yoy_anomalies else pd.DataFrame()
print(f'Within-sector YoY outliers: {len(yoy_df)}')
yoy_df.sort_values(['category', 'measure', 'year'])

Within-sector YoY outliers: 39


,year,category,measure,value,anomaly_type,detail
22,2008,Agriculture,HGF share (%),19.174,Within-sector YoY outlier (|z|>3),"Δ=+12.205, z=+3.1"
12,2008,Agriculture,JCR (%),18.467,Within-sector YoY outlier (|z|>3),"Δ=+10.564, z=+3.1"
15,2007,Agriculture,JDR (%),15.351,Within-sector YoY outlier (|z|>3),"Δ=+7.529, z=+3.1"
20,2008,Agriculture,Mean DHS growth,0.093,Within-sector YoY outlier (|z|>3),"Δ=+0.092, z=+3.2"
21,2009,Agriculture,Mean DHS growth,0.005,Within-sector YoY outlier (|z|>3),"Δ=-0.088, z=-3.1"
25,1998,Agriculture,Median productivity (£),51.000,Within-sector YoY outlier (|z|>3),"Δ=+21.000, z=+3.9"
23,2008,Agriculture,Stagnant share (%),63.817,Within-sector YoY outlier (|z|>3),"Δ=-16.957, z=-3.0"
26,2008,Business Support Services,Median productivity (£),64.000,Within-sector YoY outlier (|z|>3),"Δ=-16.000, z=-3.6"
30,1998,Business Support Services,P90/P10 dispersion,10.100,Within-sector YoY outlier (|z|>3),"Δ=-6.900, z=-3.9"
27,2004,Hospitality,Median productivity (£),40.000,Within-sector YoY outlier (|z|>3),"Δ=+7.000, z=+3.2"


In [23]:
# ── Test 3: Cross-sector outliers in each year (|z| > 3) ─────────────────────

XS_MEASURES = [
    (fd_s,   'entry_rate',      'Entry rate (%)'),
    (fd_s,   'exit_rate',       'Exit rate (%)'),
    (jf_s,   'jrr',             'JRR (%)'),
    (gr_s,   'mean_dhs_growth', 'Mean DHS growth'),
    (gc_s,   'hgf_sh',          'HGF share (%)'),
    (gc_s,   'stag_sh',         'Stagnant share (%)'),
    (prod_s, 'p50_prod',        'Median productivity (£)'),
    (prod_s, 'p90_p10',         'P90/P10 dispersion'),
]

xs_anomalies = []
for df, col, label in XS_MEASURES:
    tmp = df[['year', 'category', col]].dropna().copy()
    stats = tmp.groupby('year')[col].agg(['mean', 'std']).rename(
        columns={'mean': 'yr_mean', 'std': 'yr_std'}
    )
    tmp = tmp.merge(stats, on='year')
    tmp['z'] = (tmp[col] - tmp['yr_mean']) / tmp['yr_std']
    hits = tmp[tmp['z'].abs() > 3].copy()
    if len(hits):
        hits['measure'] = label
        hits['value']   = hits[col].round(3)
        hits['anomaly_type'] = 'Cross-sector outlier (|z|>3)'
        hits['detail'] = hits.apply(
            lambda r: f"z={r['z']:+.1f} (yr mean={r['yr_mean']:.2f}, sd={r['yr_std']:.2f})", axis=1
        )
        xs_anomalies.append(hits[['year', 'category', 'measure', 'value', 'anomaly_type', 'detail']])

xs_df = pd.concat(xs_anomalies, ignore_index=True) if xs_anomalies else pd.DataFrame()
print(f'Cross-sector outliers: {len(xs_df)}')
xs_df.sort_values(['category', 'measure', 'year'])

Cross-sector outliers: 3


,year,category,measure,value,anomaly_type,detail
1,2003,IT & Computer Services,JRR (%),85.311,Cross-sector outlier (|z|>3),"z=+3.1 (yr mean=30.19, sd=17.94)"
2,2001,Other Primary Industries,P90/P10 dispersion,63.000,Cross-sector outlier (|z|>3),"z=+3.3 (yr mean=13.36, sd=14.85)"
0,2003,Utilities,Entry rate (%),123.866,Cross-sector outlier (|z|>3),"z=+3.5 (yr mean=20.06, sd=29.38)"


In [24]:
# ── Combined anomaly table ─────────────────────────────────────────────────────

all_dfs = [df for df in [bound_df, yoy_df, xs_df] if len(df) > 0]
anomalies = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame(
    columns=['year', 'category', 'measure', 'value', 'anomaly_type', 'detail']
)

anomalies = anomalies.sort_values(['category', 'year', 'measure']).reset_index(drop=True)
anomalies.columns = ['Year', 'Sector', 'Measure', 'Value', 'Anomaly type', 'Detail']

print(f'Total anomalies flagged: {len(anomalies)}')
print(f'Unique sector-year combinations: {anomalies[["Year", "Sector"]].drop_duplicates().shape[0]}')
print()
print('Anomaly count by sector:')
print(anomalies['Sector'].value_counts().to_string())
print()
print('Anomaly count by measure:')
print(anomalies['Measure'].value_counts().to_string())

Total anomalies flagged: 43
Unique sector-year combinations: 27

Anomaly count by sector:
Sector
IT & Computer Services        8
Agriculture                   7
Utilities                     6
Other Information Services    5
Other Primary Industries      4
Transport & Logistics         3
Business Support Services     2
Hospitality                   2
Other Services                2
Manufacturing                 1
Professional Services         1
Retail Trade                  1
Social care                   1

Anomaly count by measure:
Measure
P90/P10 dispersion         10
Entry rate (%)              7
Median productivity (£)     5
JDR (%)                     5
Exit rate (%)               5
JCR (%)                     3
JRR (%)                     3
Mean DHS growth             2
Stagnant share (%)          2
HGF share (%)               1


In [25]:
# Full anomaly list — export-ready
anomalies

,Year,Sector,Measure,Value,Anomaly type,Detail
0,1998,Agriculture,Median productivity (£),51.000,Within-sector YoY outlier (|z|>3),"Δ=+21.000, z=+3.9"
1,2007,Agriculture,JDR (%),15.351,Within-sector YoY outlier (|z|>3),"Δ=+7.529, z=+3.1"
2,2008,Agriculture,HGF share (%),19.174,Within-sector YoY outlier (|z|>3),"Δ=+12.205, z=+3.1"
3,2008,Agriculture,JCR (%),18.467,Within-sector YoY outlier (|z|>3),"Δ=+10.564, z=+3.1"
4,2008,Agriculture,Mean DHS growth,0.093,Within-sector YoY outlier (|z|>3),"Δ=+0.092, z=+3.2"
5,2008,Agriculture,Stagnant share (%),63.817,Within-sector YoY outlier (|z|>3),"Δ=-16.957, z=-3.0"
6,2009,Agriculture,Mean DHS growth,0.005,Within-sector YoY outlier (|z|>3),"Δ=-0.088, z=-3.1"
7,1998,Business Support Services,P90/P10 dispersion,10.100,Within-sector YoY outlier (|z|>3),"Δ=-6.900, z=-3.9"
8,2008,Business Support Services,Median productivity (£),64.000,Within-sector YoY outlier (|z|>3),"Δ=-16.000, z=-3.6"
9,2004,Hospitality,Median productivity (£),40.000,Within-sector YoY outlier (|z|>3),"Δ=+7.000, z=+3.2"


In [26]:
# Heatmap: anomaly count by sector × year
if len(anomalies):
    heat = anomalies.groupby(['Year', 'Sector']).size().reset_index(name='count')

    alt.Chart(heat).mark_rect().encode(
        x=alt.X('Year:O', title=''),
        y=alt.Y('Sector:N', title=''),
        color=alt.Color('count:Q',
            scale=alt.Scale(scheme='orangered'),
            legend=alt.Legend(title='# anomalies')),
        tooltip=['Year:O', 'Sector:N', 'count:Q']
    ).properties(title='Anomaly count by sector and year', width=700, height=360)
else:
    print('No anomalies flagged.')

In [27]:
# Export anomaly list to Tables/
out_path = script_dir.parent / 'Tables' / 'sector_anomalies.xlsx'
anomalies.to_excel(out_path, index=False)
print(f'Exported to {out_path}')

Exported to /Users/willshepherd/shepherdwill.github.io/CEP Dynamism Project/Tables/sector_anomalies.xlsx
